# Setup

In [71]:
import json
from scipy import stats
from numpy import mean
from collections import Counter
import itertools
from numpy import nanmean, nan
import csv

def l_extend(l):
    return [lll for ll in l for lll in ll]

# CoNaLa

In [72]:
conala_models_list = ['baseline', 'tranx-annot', 'best-tranx', 'best-tranx-rerank', 'codex']

exp_metrics = ["bleu","codebleu","chrf","rougel","meteor","ruby",
               "codebertscore_f1","codebertscore_s_f1",
               "codebertscore_f3","codebertscore_s_f3",
               "gpt35_nsnr","gpt35_nswr","gpt35_wsnr","gpt35_wswr"
              ]
real_name_metrics=["BLEU","CodeBLEU","chrF","ROUGE-L","METEOR","RUBY",
                   "CodeBERTSCORE-F1 (w/o S.)","CodeBERTSCORE-F1 (w/ S.)",
                   "CodeBERTSCORE-F3 (w/o S.)","CodeBERTSCORE-F3 (w/ S.)",
                   "GPT-3.5 (w/o R.)","GPT-3.5 (w/ R.)",
                   "GPT-3.5 (w/o R.) + 0-shot-CoT","GPT-3.5 (w/ R.) + 0-shot-CoT"
                  ]
def compute(data,metric,level="example"):
    refs,preds=[],[]
    for d in data:
        refs.append([d[f"grade-{k}"] for k in conala_models_list])
        preds.append([d[f"{metric}-{k}"] for k in conala_models_list])
    if level=="example":
        return nanmean([stats.kendalltau(ref,pred).statistic for ref,pred in zip(refs,preds)]),\
                nanmean([stats.pearsonr(ref,pred).statistic for ref,pred in zip(refs,preds)]),\
                nanmean([stats.spearmanr(ref,pred).statistic for ref,pred in zip(refs,preds)])
    else:
        return stats.kendalltau(l_extend(refs),l_extend(preds)).statistic,\
            stats.pearsonr(l_extend(refs),l_extend(preds)).statistic,\
            stats.spearmanr(l_extend(refs),l_extend(preds)).statistic

In [73]:
# # Example Level
# for m,rm in zip(exp_metrics,real_name_metrics):
#     print(rm)
#     kendalls,pearsons,spearmans=[],[],[]
#     with open(f"../data/conala/conala_grade.json") as f:
#         data = json.load(f)
#     kendall,pearson,spearman = compute(data,m,level="example")
#     kendalls.append(kendall)
#     pearsons.append(pearson)
#     spearmans.append(spearman)
#     print("\t\t",
#           "{:.3f}".format(round(kendall,3))[1:],
#           "{:.3f}".format(round(pearson,3))[1:],
#           "{:.3f}".format(round(spearman,3))[1:])

In [74]:
# # Corpus Level
# for m,rm in zip(exp_metrics,real_name_metrics):
#     print(rm)
#     kendalls,pearsons,spearmans=[],[],[]
#     with open(f"../data/conala/conala_grade.json") as f:
#         data = json.load(f)
#     kendall,pearson,spearman = compute(data,m,level="corpus")
#     kendalls.append(kendall)
#     pearsons.append(pearson)
#     spearmans.append(spearman)
#     print("\t\t",
#           "{:.3f}".format(round(kendall,3))[1:],
#           "{:.3f}".format(round(pearson,3))[1:],
#           "{:.3f}".format(round(spearman,3))[1:])

# HumanEval-X

In [75]:
baseline_metrics = ["bleu","codebleu","chrf","rougel","meteor","ruby","codebertscore","gpt35"]
exp_metrics = ["bleu","codebleu","chrf","rougel","meteor","ruby",
               "codebertscore_f1","codebertscore_s_f1",
               "codebertscore_f3","codebertscore_s_f3",
               "gpt35_nsnr","gpt35_nswr"
              ]
real_name_metrics=["BLEU","CodeBLEU","chrF","ROUGE-L","METEOR","RUBY",
                   "CodeBERTSCORE-F1 (w/o S.)","CodeBERTSCORE-F1 (w/ S.)",
                   "CodeBERTSCORE-F3 (w/o S.)","CodeBERTSCORE-F3 (w/ S.)",
                   "GPT-3.5 (w/o R.)","GPT-3.5 (w/ R.)"
                  ]
def compute(data,metric,level="example"):
    refs,preds=[],[]
    for d in data:
        ks=[k.replace("grade-","") for k in d.keys() if k.startswith("grade-")]
        refs.append([d[f"grade-{k}"]["execution_custom"] for k in ks])
        preds.append([d[f"{metric}-{k}"] for k in ks])
    if level=="example":
        return nanmean([stats.kendalltau(ref,pred).statistic for ref,pred in zip(refs,preds)]),\
                nanmean([stats.pearsonr(ref,pred).statistic for ref,pred in zip(refs,preds)]),\
                nanmean([stats.spearmanr(ref,pred).statistic for ref,pred in zip(refs,preds)])
    else:
        return stats.kendalltau(l_extend(refs),l_extend(preds)).statistic,\
            stats.pearsonr(l_extend(refs),l_extend(preds)).statistic,\
            stats.spearmanr(l_extend(refs),l_extend(preds)).statistic
    
def compute_and_save_csv(data, metric, output_path):
    rows = []

    for idx, d in enumerate(data):
        # if d['task_id'] != "89":
        #     continue
        ks = [k.replace("grade-", "") for k in d.keys() if k.startswith("grade-")]

        ref = [d[f"grade-{k}"]["execution_custom"] for k in ks]
        pred = [d[f"{metric}-{k}"] for k in ks]

        # 길이가 2 이상일 때만 상관계수 계산 가능
        if len(ref) > 1 and len(pred) > 1:
            kendall = stats.kendalltau(ref, pred).statistic
            pearson = stats.pearsonr(ref, pred).statistic
            spearman = stats.spearmanr(ref, pred).statistic
        else:
            kendall = nan
            pearson = nan
            spearman = nan

        rows.append({
            "index": d["task_id"],
            "kendall": kendall,
            "pearson": pearson,
            "spearman": spearman
        })
        # print(
        #     f"index: {d['task_id']} ",
        #     f"kendall: {kendall} ",
        #     f"spearman: {spearman}"
        # )

    # CSV 파일 저장
    with open(output_path, "w", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=["index", "kendall", "pearson", "spearman"])
        writer.writeheader()
        writer.writerows(rows)

    return rows

In [76]:
def compute_other_model(data, other_model_data, level="example"):
    refs,preds=[],[]

    for d, od in zip(data, other_model_data):
        ks=[k.replace("grade-","") for k in d.keys() if k.startswith("grade-")]
        refs.append([d[f"grade-{k}"]["execution"] for k in ks])
        preds.append([
            float(od["evaluations"].get(k, 0))
            for k in ks
        ])
    if level=="example":
        return nanmean([stats.kendalltau(ref,pred).statistic for ref,pred in zip(refs,preds)]),\
                nanmean([stats.pearsonr(ref,pred).statistic for ref,pred in zip(refs,preds)]),\
                nanmean([stats.spearmanr(ref,pred).statistic for ref,pred in zip(refs,preds)])
    else:
        return stats.kendalltau(l_extend(refs),l_extend(preds)).statistic,\
            stats.pearsonr(l_extend(refs),l_extend(preds)).statistic,\
            stats.spearmanr(l_extend(refs),l_extend(preds)).statistic

In [77]:
def compute_other_model(data, other_model_data, level="example"):
    refs,preds=[],[]

    for d, od in zip(data, other_model_data):
        ks=[k.replace("grade-","") for k in d.keys() if k.startswith("grade-")]
        refs.append([d[f"grade-{k}"]["execution_custom"] for k in ks])
        preds.append([
            float(od["evaluations"].get(k, 0))
            for k in ks
        ])
    if level=="example":
        return nanmean([stats.kendalltau(ref,pred).statistic for ref,pred in zip(refs,preds)]),\
                nanmean([stats.pearsonr(ref,pred).statistic for ref,pred in zip(refs,preds)]),\
                nanmean([stats.spearmanr(ref,pred).statistic for ref,pred in zip(refs,preds)])
    else:
        return stats.kendalltau(l_extend(refs),l_extend(preds)).statistic,\
            stats.pearsonr(l_extend(refs),l_extend(preds)).statistic,\
            stats.spearmanr(l_extend(refs),l_extend(preds)).statistic

In [78]:
# Example Level
for m,rm in zip(exp_metrics,real_name_metrics):
    print(rm)
    kendalls,pearsons,spearmans=[],[],[]
    
    with open(f"../data/humaneval/humaneval_python_grade_evalplus_ratio.json") as f:
        data = json.load(f)
    kendall,pearson,spearman = compute(data,m,level="example")
    kendalls.append(kendall)
    pearsons.append(pearson)
    spearmans.append(spearman)
    print("\t","python")
    print("\t\t",
            "{:.3f}".format(round(kendall,3))[1:],
            "{:.3f}".format(round(spearman,3))[1:])

    compute_and_save_csv(data, m, output_path="correlation_results_evalplus.csv")

BLEU
	 python
		 .283 .356


/tmp/ipykernel_586416/2637996910.py:20: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  nanmean([stats.pearsonr(ref,pred).statistic for ref,pred in zip(refs,preds)]),\
/tmp/ipykernel_586416/2637996910.py:21: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  nanmean([stats.spearmanr(ref,pred).statistic for ref,pred in zip(refs,preds)])
/tmp/ipykernel_586416/2637996910.py:41: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  pearson = stats.pearsonr(ref, pred).statistic
/tmp/ipykernel_586416/2637996910.py:42: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  spearman = stats.spearmanr(ref, pred).statistic


CodeBLEU
	 python
		 .263 .328
chrF
	 python
		 .284 .357
ROUGE-L
	 python
		 .268 .334
METEOR
	 python
		 .293 .366
RUBY
	 python
		 .238 .297
CodeBERTSCORE-F1 (w/o S.)
	 python
		 .271 .343
CodeBERTSCORE-F1 (w/ S.)
	 python
		 .268 .340
CodeBERTSCORE-F3 (w/o S.)
	 python
		 .279 .351
CodeBERTSCORE-F3 (w/ S.)
	 python
		 .281 .352
GPT-3.5 (w/o R.)
	 python
		 .249 .270
GPT-3.5 (w/ R.)
	 python
		 .270 .296


In [79]:
print("Other Model")
kendalls,pearsons,spearmans=[],[],[]

with open(f"../data/humaneval/humaneval_python_grade_evalplus_ratio.json") as f:
    data = json.load(f)
with open(f"../data/humaneval/humaneval_llm_eval_gpt5_r_ref.json") as f:
    other_model_data = json.load(f)
kendall,pearson,spearman = compute_other_model(data, other_model_data, level="example")
kendalls.append(kendall)
pearsons.append(pearson)
spearmans.append(spearman)
print("\t","python")
print("\t\t",
        "{:.3f}".format(round(kendall,3))[1:],
        "{:.3f}".format(round(spearman,3))[1:])

# compute_and_save_csv(data, m, output_path="correlation_results_evalplus_qwen.csv")

Other Model
	 python
		 .410 .456


/tmp/ipykernel_586416/3445320369.py:13: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  nanmean([stats.pearsonr(ref,pred).statistic for ref,pred in zip(refs,preds)]),\
/tmp/ipykernel_586416/3445320369.py:14: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  nanmean([stats.spearmanr(ref,pred).statistic for ref,pred in zip(refs,preds)])


In [80]:
# Corpus Level
for m,rm in zip(exp_metrics,real_name_metrics):
    print(rm)
    kendalls,pearsons,spearmans=[],[],[]
    with open(f"../data/humaneval/humaneval_python_grade_evalplus_ratio.json") as f:
        data = json.load(f)
    kendall,pearson,spearman = compute(data,m,level="corpus")
    kendalls.append(kendall)
    pearsons.append(pearson)
    spearmans.append(spearman)
    print("\t","python")
    print("\t\t",
            "{:.3f}".format(round(kendall,3))[1:],
            "{:.3f}".format(round(spearman,3))[1:])

BLEU
	 python
		 .289 .393
CodeBLEU
	 python
		 .283 .385
chrF
	 python
		 .321 .436
ROUGE-L
	 python
		 .302 .409
METEOR
	 python
		 .342 .461
RUBY
	 python
		 .279 .379
CodeBERTSCORE-F1 (w/o S.)
	 python
		 .314 .426
CodeBERTSCORE-F1 (w/ S.)
	 python
		 .291 .396
CodeBERTSCORE-F3 (w/o S.)
	 python
		 .334 .452
CodeBERTSCORE-F3 (w/ S.)
	 python
		 .321 .436
GPT-3.5 (w/o R.)
	 python
		 .259 .299
GPT-3.5 (w/ R.)
	 python
		 .349 .411
